In [20]:
# 1. Safe Library Installation (Uses active environment cache)
%pip install -q transformers accelerate datasets

# 2. Prevent HuggingFace from flooding your 20GB disk space
import os
os.environ["HF_HOME"] = "/kaggle/working/hf_cache"

# 3. Hardware Verification Setup
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 System active. Current Device: {device.upper()}")
print(f"📊 Total available GPUs detected: {torch.cuda.device_count()}")

if torch.cuda.device_count() > 1:
    print("💡 Pro-Tip: You are running dual T4 GPUs. Use torch.nn.DataParallel() to maximize VRAM!")

Note: you may need to restart the kernel to use updated packages.
🚀 System active. Current Device: CPU
📊 Total available GPUs detected: 0


c:\Users\ADEGOKE\Desktop\DS-ML-AI\E-Commerce Risk And Demand Intelligent System\.kaggle-env\Scripts\python.exe: No module named pip


In [ ]:
from pathlib import Path
import pandas as pd

input_root = Path("/kaggle/input")
target_file = "nigeria_ecommerce_major_project_25000.csv"

if not input_root.exists():
    raise RuntimeError(
        "This cell must run on Kaggle, not the local VS Code kernel. "
        "Push the notebook with 'kaggle kernels push -p .' and run it on Kaggle."
    )

dataset_path = next(input_root.rglob(target_file), None)

if dataset_path is None:
    raise FileNotFoundError(
        f"{target_file} was not found under {input_root}. "
        "Confirm that the Kaggle dataset is attached to this Kernel."
    )

print(f"Using Kaggle dataset: {dataset_path}")
df = pd.read_csv(dataset_path)
df.head()

In [ ]:
# Kaggle end-to-end smoke test: GPU check, dataset analysis, and lightweight training
import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_test = torch.randn((1024, 1024), device="cuda") @ torch.randn(
        (1024, 1024), device="cuda"
    )
    torch.cuda.synchronize()
    print(f"GPU smoke test passed: {gpu_test.device}")

print(f"Dataset shape: {df.shape}")
print(f"Missing values: {int(df.isna().sum().sum())}")
print("Order-status distribution:")
status_counts = df["order_status"].value_counts(dropna=False)
print(status_counts)

status_counts.sort_values().plot(
    kind="bar",
    figsize=(8, 4),
    color="#2563eb",
    title="Order Status Distribution",
    xlabel="Order Status",
    ylabel="Number of Orders",
    rot=0,
)
plt.tight_layout()
plt.savefig("/kaggle/working/order_status_distribution.png", dpi=150)
plt.show()

target = "order_status"
drop_columns = [
    target,
    "order_id",
    "customer_id",
    "order_timestamp",
    "order_date",
]
model_df = df.dropna(subset=[target]).copy()
X = model_df.drop(
    columns=[column for column in drop_columns if column in model_df.columns]
)
y = model_df[target].astype(str)

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

numeric_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)
categorical_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)
preprocessor = ColumnTransformer(
    [
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ]
)
classifier = LogisticRegression(max_iter=200, class_weight="balanced")
pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("classifier", classifier),
    ]
)

stratify = y if y.value_counts().min() >= 2 else None
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=stratify
)
pipeline.fit(X_train, y_train)
predictions = pipeline.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f"Training rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")
print(f"Baseline accuracy: {accuracy:.4f}")
print(classification_report(y_test, predictions, zero_division=0))

summary = {
    "dataset_shape": list(df.shape),
    "missing_values": int(df.isna().sum().sum()),
    "target": target,
    "training_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "accuracy": float(accuracy),
    "cuda_available": bool(torch.cuda.is_available()),
}
with open("/kaggle/working/smoke_test_summary.json", "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=2)
pd.DataFrame({"actual": y_test.to_numpy(), "predicted": predictions}).to_csv(
    "/kaggle/working/smoke_test_predictions.csv", index=False
)
print("Saved analysis and smoke-test files in /kaggle/working/")